# Lab 1 v2 — Optimizer Benchmark (best weight theo val F1) trên FashionMNIST (Kaggle, dual T4)

**Khác v1**: best weight + early stop theo **val F1 macro** (thay vì val loss), val mỗi epoch, patience 10, max 150 epochs. Repo v1 (chọn theo loss): lab1-nhom8

Benchmark 8 optimizers × 3 LRs × 3 seeds × 2 models (MLP, CNN) = 144 runs.
**Chạy song song trên 2 GPU T4**: MLP trên GPU 0, CNN trên GPU 1.

## Setup trước khi chạy (làm 1 lần)
1. Tạo GitHub **Personal Access Token** (classic, scope `repo`) tại https://github.com/settings/tokens
2. Kaggle: **Add-ons → Secrets → Add secret**, tên `GITHUB_TOKEN`, giá trị là token vừa tạo
   (push lên GitHub luôn cần token, kể cả repo public — không có thì auto-backup bị skip)
3. Settings → Accelerator: **GPU T4 x2**
4. Settings → Bật **Internet** (cần để clone repo)

> **Resume**: session chết / Stop / hết 12h → chạy lại notebook từ đầu. Nó tự pull code, tự bỏ qua run đã xong, tự resume run dở từ checkpoint (mỗi 200 batch).

In [ ]:
import os
REPO = "https://github.com/thanh1912-ut/lab1-nhom8-v2"
WORK = "/kaggle/working/lab1-nhom8-v2"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN loaded")
except Exception:
    print("WARNING: GITHUB_TOKEN not found - git backup will be SKIPPED.")

%cd /kaggle/working
if not os.path.isdir(WORK):
    !git clone -q {REPO} lab1-nhom8-v2
else:
    !cd lab1-nhom8-v2 && git pull -q || true
%cd {WORK}
!pip install -q -r requirements.txt 2>&1 | tail -1

In [ ]:
import torch
n = torch.cuda.device_count()
print(f"torch {torch.__version__} | {n} GPU(s):",
      [torch.cuda.get_device_name(i) for i in range(n)] or "CPU only")

In [ ]:
# Pre-download dataset 1 lần (tránh 2 process download cùng lúc)
from torchvision import datasets
datasets.FashionMNIST("./data", train=True, download=True)
datasets.FashionMNIST("./data", train=False, download=True)
print("dataset ready")

## Chạy song song 2 GPU
MLP → GPU 0, CNN → GPU 1. Log mỗi process ghi ra `mlp_train.out` / `cnn_train.out`.

Cell bên dưới **không chặn** — chạy xong là return ngay, 2 process chạy nền. Sau đó có thể chạy cell xem log / TensorBoard bên dưới bất cứ lúc nào.

In [ ]:
import subprocess, os

def launch(script, gpu, logfile):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    out = open(logfile, "a")
    p = subprocess.Popen(["python", "-u", script], env=env,
                         stdout=out, stderr=subprocess.STDOUT)
    print(f"{script} -> GPU {gpu}, PID {p.pid}, log: {logfile}")
    return p

p_mlp = launch("train_mlp.py", 0, "mlp_train.out")
p_cnn = launch("train_cnn.py", 1, "cnn_train.out")

### Xem log live
- **Cell 1** — xem 1 lần 20 dòng cuối mỗi process (chạy lại để cập nhật)
- **Cell 2** — stream liên tục kiểu `tail -f`: tự refresh mỗi 5s cho đến khi hết `MINUTES` hoặc bấm nút ⏹ Interrupt. Training nền **không bị ảnh hưởng** khi interrupt.

In [ ]:
# Xem nhanh 20 dòng cuối (chạy lại cell để cập nhật)
for f in ("mlp_train.out", "cnn_train.out"):
    print(f"\n===== {f} =====")
    !tail -20 {f}

In [ ]:
# STREAM LIÊN TỤC (tail -f style) - bấm Interrupt (⏹) để thoát, train vẫn chạy nền
import time
from IPython.display import clear_output

MINUTES = 60        # thoát tự động sau N phút
LINES = 15          # số dòng hiển thị mỗi process
t_end = time.time() + MINUTES * 60
try:
    while time.time() < t_end:
        clear_output(wait=True)
        import subprocess
        for f in ("mlp_train.out", "cnn_train.out"):
            print(f"===== {f} =====")
            try:
                out = subprocess.run(["tail", "-n", str(LINES), f],
                                     capture_output=True, text=True).stdout
                print(out)
            except Exception as e:
                print(e)
        alive = {"mlp": p_mlp.poll() is None, "cnn": p_cnn.poll() is None}
        print(f"running: mlp={alive['mlp']} cnn={alive['cnn']} | "
              f"refresh 5s | Ctrl+C/⏹ to exit view")
        if not any(alive.values()):
            print("both processes finished.")
            break
        time.sleep(5)
except KeyboardInterrupt:
    print("log view stopped (training continues in background)")

### TensorBoard live
Chạy cell này rồi click link tensorboard ở output (cập nhật realtime).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/tensorboard --host 0.0.0.0

### Đợi cả 2 process xong
Cell này **chặn** đến khi cả MLP và CNN hoàn tất (hoặc bị lỗi).

In [ ]:
rc_mlp = p_mlp.wait()
rc_cnn = p_cnn.wait()
print(f"MLP exit code: {rc_mlp} | CNN exit code: {rc_cnn}")
print("(0 = xong sạch, -2/1 = bị interrupt - chạy lại cell launch để resume)")

## Push thủ công (nếu cần) + tổng quan kết quả

In [ ]:
import sys, json, glob
sys.path.insert(0, "src")
from benchmark import load_config
from gitbackup import backup
for cfg_path in ("configs/mlp.yaml", "configs/cnn.yaml"):
    backup(load_config(cfg_path), msg="manual backup from kaggle notebook")

for rp in sorted(glob.glob("outputs/results*.json")):
    r = json.load(open(rp))
    done = [h for h in r.values() if h.get("done")]
    print(f"\n{rp}: {len(done)} runs done")
    rows = sorted(done, key=lambda h: -(h.get("test") or {}).get("acc", 0))
    for h in rows[:10]:
        t = h.get("test", {})
        print(f"  {h['optimizer']:<10} lr={h['lr']:<7} s={h['seed']:<5} "
              f"test acc={t.get('acc', 0):.4f} f1={t.get('f1', 0):.4f} "
              f"best_ep={h['best_epoch']}")